01 - Data Collection & Description

## Setup
Run the cell below first. It detects whether you're in **Google Colab** or
running **locally in VS Code**, and gets the environment ready either way
(clones the repo in Colab, installs requirements, downloads NLTK data, and
adds `src/` to the path so `pipeline.py` can be imported).

In [ ]:
# ============================================================
# SETUP CELL - run this first, every time
# Works both locally (VS Code / Jupyter) and in Google Colab
# ============================================================
import os, sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO_URL = "https://github.com/Sandaru17513/NLP_Ctrl-Alt-Elite.git"
    REPO_DIR = "NLP_Ctrl-Alt-Elite"

    if not os.path.exists(REPO_DIR):
        get_ipython().system(f"git clone {REPO_URL}")
    os.chdir(f"{REPO_DIR}/notebooks")

    get_ipython().system("pip install -q -r ../requirements.txt")

    # Data files are large - if they were not committed to the repo,
    # upload them here once per Colab session.
    if not os.path.exists("../data/data.csv"):
        print("data/data.csv not found in the cloned repo.")
        print("Option A: git add + commit + push the CSVs from your")
        print("          local machine so they come down with the clone.")
        print("Option B: uncomment the lines below to upload manually.")
        # from google.colab import files
        # uploaded = files.upload()   # select data.csv + validation_dataset.csv
        # os.makedirs("../data", exist_ok=True)
        # for fname in uploaded:
        #     os.rename(fname, f"../data/{fname}")
else:
    print("Running locally (VS Code / Jupyter). Using existing .venv environment.")

import nltk
for pkg in ("punkt", "punkt_tab", "stopwords", "wordnet", "omw-1.4"):
    try:
        nltk.download(pkg, quiet=True)
    except Exception:
        pass

sys.path.append(os.path.abspath("../src"))
print("IN_COLAB =", IN_COLAB)
print("Working directory:", os.getcwd())


### 1.1 Load and inspect both datasets

In [ ]:
import pandas as pd
import numpy as np

df_train = pd.read_csv('../data/data.csv')
df_val = pd.read_csv('../data/validation_dataset.csv')

print('=== TRAINING DATASET (data.csv) ===')
print(f'Shape: {df_train.shape}')
print(f'Columns: {df_train.columns.tolist()}')
print(df_train.head())
print(df_train.dtypes)
print('Nulls per column:')
print(df_train.isnull().sum())
print(df_train['label'].value_counts())
print(df_train['label'].value_counts(normalize=True) * 100)

In [ ]:
print('=== VALIDATION DATASET (validation_dataset.csv) ===')
print(f'Shape: {df_val.shape}')
print(f'Columns: {df_val.columns.tolist()}')
print(df_val.head())
print(df_val.isnull().sum())
print(df_val['Email Type'].value_counts())

### 1.2 Standardize column names (critical step)
The two datasets use different column names. Rename them so the whole
pipeline downstream works on both without special-casing.

In [ ]:
df_train = df_train.rename(columns={'text': 'email_text', 'label': 'email_label'})

df_val = df_val.rename(columns={'Email Text': 'email_text', 'Email Type': 'email_label'})
df_val['email_label'] = df_val['email_label'].map({
    'Safe Email': 'ham',
    'Phishing Email': 'spam'
})

print('Training labels:', df_train['email_label'].unique())
print('Validation labels:', df_val['email_label'].unique())

df_train.to_csv('../data/train_renamed.csv', index=False)
df_val.to_csv('../data/val_renamed.csv', index=False)
print('Saved: train_renamed.csv, val_renamed.csv')

### 1.3 Dataset description summary (include in report)

| Attribute | data.csv (Training) | validation_dataset.csv |
|---|---|---|
| Total Records | 55,310 | 2,000 |
| Ham / Safe | 34,054 (61.6%) | 1,000 (50%) |
| Spam / Phishing | 21,256 (38.4%) | 1,000 (50%) |
| Avg Text Length | ~1,321 chars | ~87 chars |
| Missing Values | None | None |
| Usage | Train + Test (80/20 split) | Final held-out validation |

**Ethical concerns to mention in the report:**
1. Dataset bias: `data.csv` is imbalanced (~62% ham) which could bias the model toward predicting ham.
2. Privacy: real email content may contain personal information - used here for educational purposes only.
3. The validation set uses synthetic phishing emails - real-world performance may differ.
4. Misclassification risk: false positives (real emails flagged as spam) could cause real harm.